# =============================== # Modelling Notebook # ===============================

### Business Case for Predictive Modelling Task

The aim of this machine learning task is to develop a regression model that predicts house sale prices to support valuation decisions for inherited properties.

Success means achieving a low prediction error (MAE) and a strong explanatory performance (R² > 0.75), ensuring the model is reliable enough for real-world decisions.

The model outputs a predicted sale price, which is used to estimate property value and guide decisions on whether to sell or invest in renovations.

In [ ]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd().resolve()

if "jupyter" in str(BASE_DIR).lower():
    BASE_DIR = BASE_DIR.parent

SRC_PATH = BASE_DIR / "src"

sys.path.insert(0, str(SRC_PATH))

print("BASE_DIR:", BASE_DIR)
print("SRC_PATH:", SRC_PATH)
print("models exists:", (SRC_PATH / "models").exists())

## Objectives

- Fit and evaluate regression models to predict SalePrice.
- Apply feature engineering pipeline developed earlier.
- Compare candidate regression models using cross-validation.
- Select best model and optimise hyperparameters.
- Export final train/test sets, pipelines, and feature importance plot.


## Inputs 
- outputs/datasets/processed/TrainSet.csv
- outputs/datasets/processed/TestSet.csv

## Outputs
- Train set (features and target)
- Test set (features and target)
- Modeling pipeline
- Feature importance plot

### Import Cell

In [ ]:
import sys
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from models.evaluation import evaluate_regression, plot_predictions, plot_residuals
from models.model_selection import HyperparameterOptimizationSearch
from models.pipeline import build_final_pipeline

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline


---

## Change working directory

We need to set the current working directory to the parent folder for consistency.

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

## Load Cleaned Data and Split into Trained and Test Sets



In [ ]:
# Load data
train_path = BASE_DIR / "outputs/datasets/processed/TrainSet.csv"
test_path = BASE_DIR / "outputs/datasets/processed/TestSet.csv"

TrainSet = pd.read_csv(train_path)
TestSet = pd.read_csv(test_path)

target = "SalePrice"

X_train = TrainSet.drop(columns=[target])
y_train = TrainSet[target]

X_test = TestSet.drop(columns=[target])
y_test = TestSet[target]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

---

# ===============================
## Model Pipeline
# ===============================

### Model Benchmarking

#### Define Candidate Models

In [ ]:
models_quick_search = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForestRegressor": RandomForestRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=42),
    "XGBRegressor": XGBRegressor(random_state=42)
}

### Define Hyperparameter Grids (Baseline)

In [ ]:
params_quick_search = {
    "LinearRegression": {},
    "Ridge": {},
    "Lasso": {},
    "RandomForestRegressor": {},
    "GradientBoostingRegressor": {},
    "ExtraTreesRegressor": {},
    "XGBRegressor": {}
}

### Run GridSearchCV (Quick Benchmark)

#### Custom class for Hyperparameter Optimization

In [ ]:
print(X_train.select_dtypes(include="object").columns)

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_quick_search,
    params=params_quick_search
)

search.fit(
    X_train,
    y_train,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

### Evaluate Model Performance

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

grid_search_summary

### Select Best Performing Model

In [ ]:
best_model = grid_search_summary.iloc[0, 0]
best_model

### Model Selection for Hyperparameter Optimisation

The aim of this machine learning task is to build a regression model that accurately predicts house sale prices for the inherited properties and unseen data, supporting client decisions around selling, holding, or renovating.

A supervised regression approach was used, and multiple models were evaluated using cross-validation. Performance was primarily assessed using Mean Absolute Error (MAE), where lower (less negative) values indicate better predictive accuracy.

GradientBoostingRegressor achieved the best performance with a mean cross-validation score of approximately **-21,007**, outperforming other models such as RandomForestRegressor (-21,326) and Lasso (-21,972). It also demonstrated relatively low variability across folds (std ≈ 1,741), indicating stable and reliable performance.

In contrast, simpler models such as LinearRegression performed significantly worse (mean score ≈ -47,250), highlighting the importance of using more advanced ensemble methods for this dataset.

The final model outputs a predicted sale price for each property, which is used directly to estimate portfolio value and support financial decision-making for the client.

The model was trained on historical Ames housing data, including features such as living area, basement size, garage area, and year built, which were selected based on domain knowledge and prior correlation analysis.

### Define Hyperparameter Grid for Best Model

In [ ]:
models_search = {
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42)
}

params_search = {
    "GradientBoostingRegressor": {
        "model__n_estimators": [100, 300],
        "model__learning_rate": [0.1, 0.01, 0.001],
        "model__max_depth": [3, 5, 10]
    }
}

### Run GridSearchCV (Tuned Model)

In [ ]:
search = HyperparameterOptimizationSearch(
    models=models_search,
    params=params_search
)

search.fit(
    X_train,
    y_train,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

### Evaluate Tuned Model Performance

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

grid_search_summary

### Extract Best Model Pipeline

In [ ]:
best_model_name = grid_search_summary.iloc[0, 0]
pipeline_reg = grid_search_pipelines[best_model_name].best_estimator_
best_params = grid_search_pipelines[best_model_name].best_params_

pipeline_reg


## Model Evaluation 

In [ ]:
train_metrics = evaluate_regression(X_train, y_train, pipeline_reg)
test_metrics = evaluate_regression(X_test, y_test, pipeline_reg)

#### Model Train Set vs Test Set comparison 
The model achieves strong performance (R²: 0.93 train, 0.82 test) with mild overfitting. Test MAE (~$21.7k) indicates reasonable prediction error, with higher errors likely occurring on extreme property values.

---

### Plot Actual vs Predicted Values

In [ ]:
plot_predictions(X_test, y_test, pipeline_reg)

### Plot Residual Distribution

In [ ]:
plot_residuals(X_test, y_test, pipeline_reg)

#### Observations from predictions plot

The model demonstrates a strong overall fit, with predictions generally tracking actual values. However, error variance increases with property value, and the model has larger and more frequent deviations for high-priced properties, indicating reduced reliability on extreme values.

### Feature Importance

In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

final_pipeline = build_final_pipeline(
    best_params=best_params,
    numeric_features=numeric_features,
    categorical_features=categorical_features
)

final_pipeline.fit(X_train, y_train)

In [ ]:
ohe_features = final_pipeline.named_steps["preprocessor"].get_feature_names_out()
model = final_pipeline.named_steps["model"]

df_feature_importance = pd.DataFrame({
    "Feature": ohe_features,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

# Show evidence
display(df_feature_importance.head(10))


In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=df_feature_importance.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Feature Importance (Gradient Boosting)")
plt.tight_layout()
plt.show()

#### Feature Importance Summary

GrLivArea is clearly the dominant predictor, followed by YearBuilt, with TotalBsmtSF, GarageArea, and BsmtFinSF1 contributing moderately. KitchenQual_Ex has minimal impact, indicating the model is driven mainly by size and construction features.




### Final Production Pipeline

#### Define Feature Groups

#### Preprocessing Pipeline

#### Final Pipeline Model 

This pipeline uses a ColumnTransformer to apply scaling to numerical features and one-hot encoding to categorical features. This approach ensures preprocessing is performed consistently during both training and inference, improving deployment reliability and reproducibility.

#### Fit Properly and Evaluate

In [ ]:
from src.models.pipeline import build_final_pipeline
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X_train.select_dtypes(include=["object"]).columns

final_pipeline = build_final_pipeline(
    best_params=best_params,
    numeric_features=numeric_features,
    categorical_features=categorical_features
)

final_pipeline.fit(X_train, y_train)
evaluate_regression(X_test, y_test, final_pipeline)

### Push Files to Repo

#### Create Export Directory

In [ ]:
version = "v1_gb_final"
file_path = BASE_DIR / f"outputs/ml_pipeline/predict_saleprice/{version}"
os.makedirs(file_path, exist_ok=True)

#### Export Data Sets

In [ ]:
X_train.to_csv(file_path / "X_train.csv", index=False)
y_train.to_csv(file_path / "y_train.csv", index=False)

X_test.to_csv(file_path / "X_test.csv", index=False)
y_test.to_csv(file_path / "y_test.csv", index=False)

#### Export Trained Model Pipeline

In [ ]:
# Export Trained Model Pipeline
joblib.dump(final_pipeline, file_path / "model_pipeline.pkl")

# Export feature list used for training
joblib.dump(X_train.columns.tolist(), file_path / "feature_list.pkl")

### Final Model Summary

In [ ]:
final_summary = grid_search_summary.iloc[0]
final_summary